# DuckPD Event Windows and Exact Late Fusion

This self-contained notebook builds event-aligned market-reaction windows without downloading data or running a model. It demonstrates exact UTC grids, revision-safe event identity, point-in-time availability, missing-bar behavior, native reaction representations, and the difference between full-population late fusion and candidate-limited reranking.

### What you will learn
- How `DataFrame.event_windows()` applies floor and ceil anchors to an exact fixed grid.
- Why event IDs, entity keys, bar labels, and availability timestamps are explicit.
- How incomplete windows and late bars affect eligibility.
- How reaction-first retrieval composes with `embed_series()` and `search_series()`.
- Why scoring every eligible event before top-k is exact, while reranking text top-k is not.

## 1. Imports and a bounded local session

All transformations remain lazy until a displayed `collect()`. The synthetic vectors stand in for already-produced news embeddings; this notebook does not introduce or imply a joint text/time-series model.

In [ ]:
import json

import pandas as pandas

import duckpd as pd

session = pd.connect(memory_limit="1GB", threads=2)
session

## 2. Create complete bars and revisioned events

`bar_start` labels the start of a complete one-minute bar. Most bars become available at their end. One B bar is deliberately published seven minutes late. Event observation 10 has two upstream revisions, so `(event_observation_id, revision)` is the event identity; `event_version_key` is a stable scalar top-k tie-breaker.

In [ ]:
observations = session.sql(
    """
    SELECT
        symbol,
        TIMESTAMP '2024-01-01 12:00:00' + step * INTERVAL 1 MINUTE AS bar_start,
        TIMESTAMP '2024-01-01 12:00:00' + step * INTERVAL 1 MINUTE
            + INTERVAL 1 MINUTE
            + CASE WHEN symbol = 'B' AND step = 2
                   THEN INTERVAL 7 MINUTE ELSE INTERVAL 0 MINUTE END
          AS bar_available_at,
        scale * (step + 1)::DOUBLE AS return_1m
    FROM (VALUES ('A', 1.0), ('B', 10.0)) AS entities(symbol, scale)
    CROSS JOIN range(7) AS steps(step)
    ORDER BY symbol, bar_start
    """
)

events = session.sql(
    """
    SELECT * FROM (VALUES
      (10, 1, 101, 'A', TIMESTAMP '2024-01-01 12:02:30',
       TIMESTAMP '2024-01-01 12:02:31', 'Initial A report'),
      (10, 2, 102, 'A', TIMESTAMP '2024-01-01 12:02:30',
       TIMESTAMP '2024-01-01 12:04:00', 'Corrected A report'),
      (20, 1, 201, 'B', TIMESTAMP '2024-01-01 12:02:30',
       TIMESTAMP '2024-01-01 12:02:45', 'B report'),
      (99, 1, 991, 'A', TIMESTAMP '2024-01-01 12:02:30',
       TIMESTAMP '2024-01-01 12:02:31', 'Query event'),
      (30, 1, 301, 'A', TIMESTAMP '2024-01-01 12:08:30',
       TIMESTAMP '2024-01-01 12:08:31', 'Incomplete future event')
    ) AS t(
      event_observation_id, revision, event_version_key, symbol, event_time,
      event_available_at, headline
    )
    ORDER BY event_version_key
    """
)

news_embeddings = session.sql(
    """
    SELECT * FROM (VALUES
      (101, [0.0, 1.0]::FLOAT[2]),
      (102, [0.2, 0.8]::FLOAT[2]),
      (201, [1.0, 0.0]::FLOAT[2]),
      (991, [1.0, 0.0]::FLOAT[2]),
      (301, [0.5, 0.5]::FLOAT[2])
    ) AS t(event_version_key, news_vector)
    """
)

print(observations)
print(events)
print(news_embeddings)
print("Executions so far:", session.execution_count)

## 3. Build exact pre-event windows

For `window=(-2, 1)` and `anchor="floor"`, an event at 12:02:30 anchors at 12:02 and requests exactly 12:00, 12:01, and 12:02. This is not the next three available bars. `bar_label="start"` asserts the source convention, and both availability columns participate in the result contract.

In [ ]:
reaction_windows = observations.event_windows(
    events,
    on="bar_start",
    bar_label="start",
    event_on="event_time",
    by="symbol",
    event_id=("event_observation_id", "revision"),
    columns={"return_window": "return_1m"},
    window=(-2, 1),
    step="PT1M",
    anchor="floor",
    available_at="bar_available_at",
    event_available_at="event_available_at",
    incomplete="null",
    metadata_prefix="reaction",
)
event_operation = json.loads(reaction_windows.explain(mode="json"))["execution_boundaries"][
    "embedding_operations"
][0]
reactions = reaction_windows.merge(
    news_embeddings,
    on="event_version_key",
    how="left",
    validate="one_to_one",
)
event_operation

## 4. Inspect ordering, completeness, and availability

Arrays are ordered by grid offset. Event 30 lacks two required bars, so its array and complete-window availability are whole nulls. B's complete array is retained, but its late bar moves `reaction_window_available_at` to 12:10. Revision 2 cannot become available before the corrected event row is known.

In [ ]:
reaction_preview = reactions[
    [
        "event_observation_id",
        "revision",
        "symbol",
        "return_window",
        "reaction_window_start",
        "reaction_window_end",
        "reaction_window_count",
        "reaction_window_complete",
        "reaction_window_available_at",
    ]
].collect()
reaction_preview

## 5. Compare floor and ceil at a non-grid event time

With offsets `(-1, 2)`, floor requests 12:01–12:03 while ceil requests 12:02–12:04. At an exact minute, both anchors choose that minute.

In [ ]:
first_revision = events[(events["event_observation_id"] == 10) & (events["revision"] == 1)]


def anchored_window(anchor):
    return observations.event_windows(
        first_revision,
        on="bar_start",
        bar_label="start",
        event_on="event_time",
        by="symbol",
        event_id=("event_observation_id", "revision"),
        columns={"return_window": "return_1m"},
        window=(-1, 2),
        step="PT1M",
        anchor=anchor,
        available_at="bar_available_at",
        event_available_at="event_available_at",
    )[["return_window", "window_window_start", "window_window_end"]].collect()


pandas.concat(
    {"floor": anchored_window("floor"), "ceil": anchored_window("ceil")},
    names=["anchor"],
)

## 6. Apply a point-in-time eligibility cutoff after extraction

The cutoff is an output eligibility rule. DuckPD keeps it above the event-window optimizer barrier, so it cannot truncate contributing source bars. B is unavailable at 12:05 because one required bar arrived at 12:10.

In [ ]:
known_by_1205 = reactions[
    reactions["reaction_window_complete"]
    & (reactions["reaction_window_available_at"] <= pandas.Timestamp("2024-01-01 12:05:00"))
]
known_by_1205[
    ["event_observation_id", "revision", "symbol", "reaction_window_available_at"]
].collect()

## 7. Encode complete reactions in a declared fixed-grid space

The representation must agree with the event-window length, cadence, and channel meaning. Native encoding remains inside DuckDB.

In [ ]:
reaction_space = pd.series_representation(
    window=3,
    channels=("simple_return",),
    sampling="fixed_grid",
    step="PT1M",
    data_contract="demo/simple-return/v1",
    normalization="none",
)

event_bank = reactions[reactions["reaction_window_complete"]].embed_series(
    columns={"simple_return": "return_window"},
    into="reaction_vector",
    representation=reaction_space,
    null_policy="error",
)

json.loads(event_bank.explain(mode="json"))["execution_boundaries"]["embedding_operations"]

## 8. Reaction-first exact retrieval

Search the reaction space first, then retain the stable event keys needed to inspect or join news. The query event itself and superseded revisions are excluded before ranking.

In [ ]:
eligible_versions = event_bank[
    (event_bank["event_observation_id"] != 99) & (event_bank["revision"] == 1)
]
reaction_matches = eligible_versions.vector.search_series(
    {"simple_return": [1.0, 2.0, 3.0]},
    column="reaction_vector",
    representation=reaction_space,
    metric="l2",
    k=2,
    tie_breaker="event_version_key",
)
reaction_matches[["event_observation_id", "revision", "headline", "_distance"]].collect()

## 9. Score the full eligible population for exact late fusion

A synthetic news query favors B, while the reaction query strongly favors A. Exact late fusion computes both distances for every eligible event, combines them under an explicit example rule, excludes the query event, and only then applies top-k. The weights are not probabilities or a universal calibration.

In [ ]:
reaction_query = session.embed_series_query(
    {"simple_return": [1.0, 2.0, 3.0]},
    representation=reaction_space,
)

eligible = event_bank[
    (event_bank["revision"] == 1)
    & (event_bank["event_observation_id"] != 99)
    & (event_bank["reaction_window_available_at"] <= pandas.Timestamp("2024-01-01 12:15:00"))
    & event_bank["news_vector"].notna()
    & event_bank["reaction_vector"].notna()
]

scored = eligible.assign(
    text_distance=lambda frame: frame["news_vector"].vector.distance([1.0, 0.0], metric="l2"),
    reaction_distance=lambda frame: frame["reaction_vector"].vector.distance(
        reaction_query, metric="l2"
    ),
)
scored = scored.assign(
    combined_distance=lambda frame: 0.4 * frame["text_distance"] + 0.6 * frame["reaction_distance"]
)
exact_fusion = scored.sort_values("event_version_key").nsmallest(
    2, ["combined_distance", "event_version_key"]
)
exact_result = exact_fusion[
    [
        "event_observation_id",
        "headline",
        "text_distance",
        "reaction_distance",
        "combined_distance",
    ]
].collect()
exact_result

## 10. Contrast candidate-limited text-first reranking

Taking text top-1 first produces only B. No later reaction score can recover A, even though A is the global winner under the combined rule. Candidate-limited reranking can be useful for latency, but it must not be labeled exact late fusion.

In [ ]:
text_limited = eligible.vector.search(
    [1.0, 0.0],
    column="news_vector",
    metric="l2",
    k=1,
    tie_breaker="event_version_key",
)
text_limited_result = text_limited[["event_observation_id", "headline", "_distance"]].collect()

assert exact_result.iloc[0]["event_observation_id"] == 10
assert text_limited_result.iloc[0]["event_observation_id"] == 20
pandas.concat(
    {"exact_full_population": exact_result.head(1), "text_top_1": text_limited_result},
    names=["workflow"],
)

## 11. Make overlap exclusion explicit

If the query event remains eligible, it trivially wins because both synthetic vectors match. Event and temporal overlap exclusions are application decisions, not assumptions DuckPD can invent.

In [ ]:
with_query = event_bank[event_bank["revision"] == 1].assign(
    text_distance=lambda frame: frame["news_vector"].vector.distance([1.0, 0.0], metric="l2"),
    reaction_distance=lambda frame: frame["reaction_vector"].vector.distance(
        reaction_query, metric="l2"
    ),
)
with_query = with_query.assign(
    combined_distance=lambda frame: 0.4 * frame["text_distance"] + 0.6 * frame["reaction_distance"]
)
including_query = (
    with_query.sort_values("event_version_key")
    .nsmallest(1, ["combined_distance", "event_version_key"])[
        ["event_observation_id", "headline", "combined_distance"]
    ]
    .collect()
)
assert including_query.iloc[0]["event_observation_id"] == 99
including_query

## 12. Production checklist

- Version event and bar revisions upstream when reconstructing historical knowledge.
- Keep occurrence time, first-known time, and artifact production time distinct.
- Use complete, start-labeled bars on a declared UTC cadence.
- Treat missing slots as missing data, never as permission to shift the grid.
- Apply availability and modality eligibility before exact scoring.
- Exclude the query event and application-defined overlapping observations.
- Record score calibration, weights, missing-modality policy, and their versions.
- Label bounded first-stage reranking as candidate-limited rather than globally exact.

In [ ]:
print("Total explicit DuckPD executions:", session.execution_count)
session.close()
print("Event-window walkthrough complete.")